# Working with files and batch analysis

Adapted from the previous files lecture and its batch-analysis revision. Keep the `data` folder beside this notebook. Start Jupyter in this folder. Lecture questions are for discussion; the exercises contain the independent practice.

## From one file to a dataset
Experimental data often live in one file per animal, day, or condition. Our workflow is:
1. Inspect and load one file.
2. Compute a result and check it.
3. Find the other files and repeat the same analysis.
4. Save a summary with enough information to identify each row.

The demonstration uses two invented animals, each tested on five trials before and after training.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from glob import glob
import os

## Paths and the working directory
A relative path starts at the notebook's working directory. An absolute path starts at the filesystem root. `data/demo_sessions/mouse_A/initial.csv` means: enter `data`, then `demo_sessions`, then `mouse_A`, and open `initial.csv`.

Run this cell and inspect the returned location. If loading fails, check the working directory and whether you extracted the whole folder.

In [ ]:
print(os.getcwd())
print(os.path.exists('data/demo_sessions/mouse_A/initial.csv'))

## Load one text file
Open `data/demo_sessions/mouse_A/initial.csv` in a text editor. Its first line is `trial,correct`; the following rows contain trial numbers and 0/1 outcomes. A comma separates columns (the **delimiter**).

`skiprows=1` skips the header. `np.loadtxt` returns an array. **Predict:** what will its shape be?

In [ ]:
data = np.loadtxt('data/demo_sessions/mouse_A/initial.csv', delimiter=',', skiprows=1)
print(data)
print(data.shape)

Each row is a trial; column 0 is the trial number and column 1 is the outcome. Average the outcomes, not both columns.

**Ask:** why would `np.mean(data)` give the wrong answer? **Reverse:** which column would you select to print trial numbers?

In [ ]:
accuracy = np.mean(data[:, 1])
print('Accuracy:', accuracy)

## Load an Excel workbook
An Excel workbook can contain several named sheets. `data/demo_mouse_A.xlsx` contains `initial` and `final` sheets with the same invented mouse A trials as our CSV files. Each sheet has column labels `trial` and `correct`.

Use pandas to read a sheet into a **DataFrame**: a table with row and column labels. The first row supplies the column names by default. For this `.xlsx` example, pandas uses the `openpyxl` package. If needed, install `pandas` and `openpyxl` into your notebook's Python environment before class (for example, `uv pip install pandas openpyxl` in its activated environment).

In JupyterLite, the next cell installs the Excel reader automatically; the first run needs an internet connection.

**Predict:** how many rows and columns should the `initial` sheet contain?

In [ ]:
import sys

if sys.platform == "emscripten":  # JupyterLite runs Python in the browser.
    import piplite
    await piplite.install("openpyxl")

import pandas as pd

excel_table = pd.read_excel('data/demo_mouse_A.xlsx', sheet_name='initial', engine='openpyxl')
print(excel_table)
print(excel_table.columns)
print(type(excel_table))

Convert the numeric table with `.to_numpy()` to use the array indexing we already know. The column labels are not part of the resulting array: column 0 is `trial` and column 1 is `correct`.

**Ask:** why must we check the column order before selecting column 1? **Reverse:** load the `final` sheet and calculate its accuracy.

In [ ]:
excel_values = excel_table.to_numpy()
print(excel_values.shape)
print('Accuracy:', np.mean(excel_values[:, 1]))

## Load a MATLAB file
A `.mat` file can store several named variables. The supplied `data/demo_mouse_A.mat` contains `initial` and `final`, each a five-row, two-column array with trial number and outcome. These are the same invented trials as in the Excel workbook and CSV files.

`scipy.io.loadmat` returns a dictionary. Inspect its keys, then retrieve the array by its variable name. Keys such as `__header__` describe the file; they are not measurements. SciPy is already used elsewhere in Week 4.

**Predict:** will loading the file return an array directly, or a collection containing arrays?

In [ ]:
from scipy.io import loadmat

matlab_data = loadmat('data/demo_mouse_A.mat')
print(matlab_data.keys())
matlab_values = matlab_data['initial']
print(matlab_values)
print(matlab_values.shape)
print('Accuracy:', np.mean(matlab_values[:, 1]))

The loaded arrays use Python's zero-based indexing: the second column is `[:, 1]`. `loadmat` normally preserves two-dimensional row/column vectors, so inspect `.shape` before indexing.

**Ask:** what is the difference between `matlab_data['initial']` and `matlab_values[:, 1]`? **Reverse:** retrieve `final` and compute its accuracy.

**File-version note:** this example uses the MAT v5 format. SciPy reads MATLAB formats through v7.2; v7.3 files use HDF5 and need an HDF5-aware reader such as `h5py`. MATLAB structs and cell arrays require additional interpretation, so the example deliberately uses numeric matrices.

Reference: [pandas.read_excel](https://pandas.pydata.org/docs/reference/api/pandas.read_excel.html) and [scipy.io.loadmat](https://docs.scipy.org/doc/scipy/reference/generated/scipy.io.loadmat.html).

## Discover files with glob
`glob` finds paths matching a pattern. `*` matches any sequence of characters, and `?` matches one character. Results are not guaranteed to arrive in a useful order; `sorted` makes the order predictable.

**Predict:** which of these patterns finds only final-session files?

In [ ]:
print(sorted(glob('data/demo_sessions/*')))
print(sorted(glob('data/demo_sessions/*/*.csv')))
print(sorted(glob('data/demo_sessions/*/final.csv')))

## Repeat the same analysis
Start with an empty list, load each file, select its outcome column, and collect one mean per file. Here index-based iteration follows the earlier loop lessons.

**Ask:** how many results should there be? **Reverse:** change the pattern to analyze the initial sessions.

In [ ]:
data_files = sorted(glob('data/demo_sessions/*/final.csv'))
print(data_files)
results = []
for index in range(len(data_files)):
    data = np.loadtxt(data_files[index], delimiter=',', skiprows=1)
    results.append(np.mean(data[:, 1]))
print(results)

## Split and assemble paths
Use `os.path` rather than splitting strings at `/`: the path separator can differ between operating systems. A filename has a **stem** and an **extension**. A path also identifies its parent folder.

In [ ]:
path = data_files[0]
folder = os.path.dirname(path)
filename = os.path.basename(path)
stem, extension = os.path.splitext(filename)
animal = os.path.basename(folder)
output_path = os.path.join('results', animal, stem + '_summary.txt')
print(folder, filename, stem, extension)
print(output_path)

**Ask:** which part changes if the input is `initial.csv`? **Reverse:** construct a path with the same animal folder but filename `accuracy.csv`.

## Keep related measurements together
Never assume that two separate file lists are correctly paired by position. Discover each animal's folder, then build both filenames from that folder. This preserves animal identity. A missing file then gives a useful error instead of silently shifting the pairs.

In [ ]:
animal_folders = sorted(glob('data/demo_sessions/*/'))
summary = []
animal_names = []
for index in range(len(animal_folders)):
    folder = animal_folders[index]
    initial = np.loadtxt(os.path.join(folder, 'initial.csv'), delimiter=',', skiprows=1)
    final = np.loadtxt(os.path.join(folder, 'final.csv'), delimiter=',', skiprows=1)
    summary.append([np.mean(initial[:, 1]), np.mean(final[:, 1])])
    animal_names.append(os.path.basename(os.path.normpath(folder)))
summary = np.array(summary)
print(animal_names)
print(summary)

## Save, reload, and check
`os.makedirs(..., exist_ok=True)` creates a folder, including missing parent folders. `np.savetxt` saves a numeric table. Here the header records the row order.

Running these cells again replaces the generated files inside `results`; the input data are kept separately.

In [ ]:
os.makedirs('results', exist_ok=True)
header = 'row order: ' + ', '.join(animal_names) + '\ninitial_accuracy,final_accuracy'
np.savetxt('results/demo_summary.csv', summary, delimiter=',', header=header)
loaded = np.loadtxt('results/demo_summary.csv', delimiter=',')
print(summary, loaded)
plt.plot([0, 1], loaded.T, 'o-')
plt.xticks([0, 1], ['Initial', 'Final'])
plt.ylabel('Accuracy')
plt.ylim(0, 1)
plt.savefig('results/demo_accuracy.png', bbox_inches='tight')
plt.show()

`np.savetxt` starts header lines with `#` by default; `np.loadtxt` skips those comment lines.

**Ask:** why must a summary record which animal each row describes?

## NumPy files: arrays and named collections
`.npy` stores one array; `.npz` stores several named arrays. Unlike a text table, these formats preserve array shapes and numeric types. The `.npz` result is dictionary-like; use a key to retrieve an array.

In [ ]:
np.save('results/accuracy.npy', summary)
print(np.load('results/accuracy.npy'))
np.savez('results/accuracy.npz', scores=summary, animals=np.array(animal_names))
archive = np.load('results/accuracy.npz')
print(archive.files)
print(archive['animals'])
print(archive['scores'])
archive.close()